In [2]:
import os
import torch
from transformers import BertTokenizer, BertForSequenceClassification
from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding
from tqdm import tqdm

# 设置环境变量（根据你的实际情况调整）
os.environ['http_proxy'] = 'http://127.0.0.1:7890'
os.environ['https_proxy'] = 'http://127.0.0.1:7890'
os.environ['all_proxy'] = 'socks5://127.0.0.1:7890'
os.environ['HF_HOME'] = '/home/bobsun/cambrige/MedMinist/hugginface'
os.environ['HUGGINGFACE_HUB_CACHE'] = '/home/bobsun/cambrige/MedMinist/hugginface'
os.chdir("/home/bobsun/cambrige/MedMinist/Transformer")
os.system("pwd")

# 加载 AG News 数据集
dataset = load_dataset('ag_news')

# 加载 BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')


def preprocess_function(examples):
    tokenized = tokenizer(examples['text'], padding='max_length', truncation=True, max_length=71)
    return {
        'input_ids': tokenized['input_ids'],
        'attention_mask': tokenized['attention_mask'],
        'labels': examples['label']
    }


tokenized_datasets = dataset.map(preprocess_function, batched=True, remove_columns=["text"])

# 定义 DataCollator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, padding='max_length', max_length=71, return_tensors='pt')

# 创建 DataLoader
train_dataset = tokenized_datasets["train"]
test_dataset = tokenized_datasets["test"]

train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=data_collator)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=data_collator)

# 加载预训练的 BERT 模型并加载训练好的权重
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=4)
model.load_state_dict(torch.load('bert_ag_news.pth', map_location='cpu'))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
model.eval()

/home/bobsun/cambrige/MedMinist/Transformer


train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [7]:
import shap
import numpy as np
from IPython.display import display

# 定义一个用于分类的 pipeline
# 注意：如果使用 GPU，请确保 'device' 参数设置为 0
classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    top_k=None,  # 替代 return_all_scores=True
    device=0 if torch.cuda.is_available() else -1
)

# 创建一个 SHAP 的 TransformersPipeline 包装器
# 这里 rescale_to_logits 设为 False，因为我们希望解释的是概率
pmodel = shap.models.TransformersPipeline(classifier, rescale_to_logits=False)

# 定义 masker，移除 padding=True 参数
masker = shap.maskers.Text(tokenizer)

# 创建 SHAP 解释器
# 对于多分类问题，SHAP 会为每个类别生成独立的 SHAP 值
explainer = shap.Explainer(pmodel, masker)

In [29]:
from IPython.display import display
import matplotlib.pyplot as plt

# 定义要解释的文本样本
# text = "Michael Phelps won the gold medal in the 400 individual medley and set a world record in a time of 4 minutes 8.26 seconds."
# text = "TGn Sync Proposes New WLAN Standard	The battle over home entertainment networking is heating up as a coalition proposes yet another standard for the IEEE #39;s consideration."
text = "LONDON (Reuters) - Oil prices surged to a new high of \$47 a  barrel on Wednesday after a new threat by rebel militia against  Iraqi oil facilities and as the United States said inflation  had stayed in check despite rising energy costs."
# 获取类别标签

labels = classifier.model.config.id2label  # 例如 {0: 'LABEL_0', 1: 'LABEL_1', 2: 'LABEL_2', 3: 'LABEL_3'}

# 生成 SHAP 值
shap_values = explainer([text])  # 确保传递的是列表

# 查看模型预测
predictions = classifier([text])  # 确保传递的是列表
print(f"Model Predictions: {predictions}")

# 找到得分最高的预测类别
if isinstance(predictions, list) and len(predictions) > 0 and isinstance(predictions[0], list):
    # 找到得分最高的类别
    prediction = max(predictions[0], key=lambda x: x['score'])  # predictions[0] 是第一个文本的预测列表
    print(f"Model Prediction: {prediction['label']} with score {prediction['score']:.4f}")

    # 提取预测类别的索引
    predicted_label = prediction['label']
    try:
        target_class = int(predicted_label.split('_')[-1])
    except ValueError:
        raise ValueError(f"Unexpected label format: {predicted_label}. Expected format 'LABEL_x'.")

    print(f"Target Class Index: {target_class}")

    # 提取目标类别的 SHAP 值
    # shap_values.values 的形状通常是 (samples, features, classes)
    # 对于单个样本，形状为 (1, features, classes)
    shap_values_target = shap_values.values[0, :, target_class]  # shape: (features,)
    base_values_target = shap_values.base_values[0, target_class]  # scalar

    # 创建新的 SHAP 解释对象，仅包含目标类别的 SHAP 值
    shap_values_filtered = shap.Explanation(
        values=shap_values_target,
        base_values=base_values_target,
        data=shap_values.data[0],
        feature_names=shap_values.feature_names
        # display_data=shap_values.data[0]  # 可选，根据 SHAP 版本决定是否需要
    )

    # 可视化 SHAP 值
    shap.plots.text(shap_values_filtered, display=labels[target_class])
else:
    print("No predictions returned by the classifier or unexpected format.")

Model Predictions: [[{'label': 'LABEL_2', 'score': 0.999075174331665}, {'label': 'LABEL_0', 'score': 0.0005752071738243103}, {'label': 'LABEL_1', 'score': 0.00023471158056054264}, {'label': 'LABEL_3', 'score': 0.000114946007670369}]]
Model Prediction: LABEL_2 with score 0.9991
Target Class Index: 2


In [30]:


inputs = encode_text(text)
input_ids = inputs['input_ids'].to(device)
attention_mask = inputs['attention_mask'].to(device)


attention_mask = attention_mask.squeeze().detach().cpu().numpy()
zero_indices = np.where(attention_mask == 0)[0]

result = shap_values_target
result_min = np.min(result)
result_max = np.max(result)

if zero_indices.size > 0:
    first_zero_index = zero_indices[0]
    result = result[:first_zero_index]

result_norm = (result - result_min) / (result_max - result_min)
tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
display(render_text_with_saliency(tokens, result_norm))

latex = saliency_to_latex(tokens, result_norm)

The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.


The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.


In [24]:
from matplotlib import cm, colors

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import matplotlib.cm as cm
from IPython.display import display, HTML
from matplotlib.cm import get_cmap
def saliency_to_latex(tokens, saliency, cmap='viridis'):
    """
    将tokens和saliency scores转换为带有背景颜色和透明度的LaTeX代码，使用RGB颜色模型。

    Args:
        tokens (list): 分词后的token列表。
        saliency (numpy array): 对应每个token的归一化saliency scores（0到1）。
        cmap (str): 使用的colormap名称（默认是'viridis'）。

    Returns:
        str: 生成的LaTeX代码。
    """
    # 选择colormap
    colormap = cm.get_cmap(cmap)

    # 归一化saliency scores
    norm = colors.Normalize(vmin=0, vmax=1)

    latex_tokens = []
    for token, score in zip(tokens, saliency):
        # 获取颜色
        rgba = colormap(norm(score))
        # 提取RGB值（忽略alpha通道）
        r, g, b, _ = rgba
        # 计算透明度，映射到0.4到1.0之间
        opacity = 0.4 + 0.6 * score
        # 生成带有RGB背景颜色和透明度的TikZ命令
        latex_token = (
            f'\\begin{{tikzpicture}}[baseline=(word.base)]\n'
            f'  \\node[fill={{rgb,1:red,{r:.2f}; green,{g:.2f}; blue,{b:.2f}}}, '
            f'opacity={opacity:.2f}, inner sep=1pt, rounded corners=2pt] (word) '
            f'{{\\strut {token}}};\n'
            f'\\end{{tikzpicture}}'
        )
        latex_tokens.append(latex_token)

    # 将所有token连接为一行
    latex_line = ' '.join(latex_tokens)
    return latex_line

def bert_cam_one_layer(layer_idx,class_idx):
    layers = model.encoder.layer
    activation = layers[layer_idx].deb.activation.squeeze(0).cpu().detach().numpy() # [70,768]

    W = layers[layer_idx].deb.W.weight
    pred_fc = layers[layer_idx].deb.pred_fc.weight
    weight = pred_fc @ W

    neuron = weight[class_idx].cpu().detach().numpy() # ndarray: [768] 单个神经元的权重, 代表70(max_length = 70) 个768维向量,求平均以后对该类别做的贡献

    a = neuron * activation
    sailency = np.sum(a, axis=1)
    return sailency

def get_all_sailency(model,input_ids,attention_mask):
    logits = model(input_ids, attention_mask=attention_mask)
    print("distribution:",torch.nn.Softmax(dim=1)(logits))
    predictions = torch.argmax(logits, dim=-1)
    sailencies = []
    for i in range(12):
        sailency = bert_cam_one_layer(i,predictions.item())
        sailencies.append(sailency)
    sailencies = np.array(sailencies)
    return sailencies

def encode_text(text):
    return tokenizer(text, padding='max_length', truncation=True, max_length=71, return_tensors='pt')


def render_text_with_saliency(tokens, saliency, cmap='Reds'):
    """
    Render text with tokens colored according to saliency scores.

    Args:
        tokens (list): List of tokens.
        saliency (numpy array): Array of saliency scores corresponding to tokens.
        cmap (str): Colormap to use.

    Returns:
        HTML object displaying colored text.
    """
    # 归一化saliency scores到[0,1]
    norm = Normalize(vmin=np.min(saliency), vmax=np.max(saliency))
    cmap = cm.get_cmap(cmap)

    # 将saliency转换为颜色
    cmap = get_cmap('viridis')

    # 将每个值映射为颜色
    colors = [plt.cm.colors.rgb2hex(cmap(norm(score))) for score in saliency]

    # 生成带有颜色的HTML文本
    colored_tokens = [
        f'<span style="background-color:{color}; padding:2px; border-radius:3px;">{token}</span>'
        for token, color in zip(tokens, colors)
    ]

    # 连接tokens
    html_text = ' '.join(colored_tokens)

    return HTML(html_text)
